In [ ]:
# %conda install statsmodels

2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.11.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /opt/anaconda3/envs/CompPrograming250

  added / updated specs:
    - statsmodels


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    patsy-1.0.2                |  py313hca03da5_0         519 KB
    statsmodels-0.14.6         |  py313haa24f5a_0        11.1 MB
    ------------------------------------------------------------
                                           Total:        11.6 MB

The following NEW packages will be INSTALLED:

  patsy              pkgs/main/osx-arm64::patsy-1.0.2-py313hca03da5_0 
  statsmodels        pkgs/main/

In [1]:
import pandas as pd
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df = pd.read_csv("./data/penguins_cleaned.csv")
df.head(1)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181,3750,male


In [2]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()}")

species: 3
island: 3
bill_length_mm: 163
bill_depth_mm: 79
flipper_length_mm: 54
body_mass_g: 93
sex: 2


In [3]:
df["sex"].unique()

array(['male', 'female'], dtype=object)

In [5]:
males = df.loc[df["sex"] == "male", "body_mass_g"]
females = df.loc[df["sex"] == "female", "body_mass_g"]
t, p = stats.ttest_ind(males, females)
print(f"t={t:.4f}, p={p:.4f}")

t=8.5417, p=0.0000


In [9]:
deg_f = males.count() + len(females) - 2
alpha = 0.05
crit_t = stats.t.ppf(1-alpha/2, deg_f)
print(f"t = {t:.4f}, critical t = {crit_t:.4f}")

t = 8.5417, critical t = 1.9672


In [10]:
males.mean() - females.mean()

np.float64(683.4117965367964)

In [11]:
df["species"].unique()

array(['Adelie', 'Gentoo', 'Chinstrap'], dtype=object)

In [ ]:
adelie = df.loc[df["species"] == "Adelie", "body_mass_g"]
gentoo = df.loc[df["species"] == "Gentoo", "body_mass_g"]
chinstrap = df.loc[df["species"] == "Chinstrap", "body_mass_g"]

f, p = stats.f_oneway(adelie, gentoo, chinstrap)
print(f"f={f:.4f}, p={p:.4f}")

f=341.8949, p=0.0000


In [17]:
cats = df["species"].unique()
samples = [df.loc[df["species"] == c, "body_mass_g"] for c in cats]
f, p = stats.f_oneway(*samples)
print(f"f={f:.4f}, p={ p:.4f}")

f=341.8949, p=0.0000


In [18]:
dfb = len(cats) - 1
dfw = len(df["body_mass_g"]) - len(cats)
alpha = 0.05
crit_f = stats.f.ppf(1-alpha, dfb, dfw)
print(f"f = {f:.4f}, critical f = {crit_f:.4f}")

f = 341.8949, critical f = 3.0231


In [20]:
scipy_tukey = stats.tukey_hsd(*samples)
print(scipy_tukey)
cats

Pairwise Group Comparisons (95.0% Confidence Interval)
Comparison  Statistic  p-value  Lower CI  Upper CI
 (0 - 1)  -1386.273     0.000 -1520.255 -1252.290
 (0 - 2)    -26.924     0.916  -186.201   132.353
 (1 - 0)   1386.273     0.000  1252.290  1520.255
 (1 - 2)   1359.349     0.000  1194.430  1524.267
 (2 - 0)     26.924     0.916  -132.353   186.201
 (2 - 1)  -1359.349     0.000 -1524.267 -1194.430



array(['Adelie', 'Gentoo', 'Chinstrap'], dtype=object)

In [22]:
sm_tukey = pairwise_tukeyhsd(
    endog=df["body_mass_g"],
    groups=df["species"],
    alpha=alpha
)
sm_tukey.summary()

group1,group2,meandiff,p-adj,lower,upper,reject
Adelie,Chinstrap,26.9239,0.9164,-132.3528,186.2005,False
Adelie,Gentoo,1386.2726,0.0,1252.2897,1520.2554,True
Chinstrap,Gentoo,1359.3487,0.0,1194.4304,1524.2671,True


In [23]:
gentoo = df.loc[df["species"] == "Gentoo", "body_mass_g"]
adelie = df.loc[df["species"] == "Adelie", "body_mass_g"]
gentoo.mean() - adelie.mean()

np.float64(1386.2725912282726)